# Transformaciones Flexibles con el Método `.apply()`

## 🎯 Objetivos
Pandas ofrece funciones optimizadas para la mayoría de las tareas, pero a veces necesitamos aplicar una lógica personalizada que no existe de forma nativa. El método `.apply()` es la herramienta definitiva para estos casos. En este notebook aprenderás a:
1. Aplicar funciones predefinidas (como las de NumPy) a una Serie de datos.
2. Crear y aplicar tus propias funciones personalizadas a todo un DataFrame.
3. Controlar la dirección de la operación mediante el parámetro `axis`.
4. Entender la diferencia de rendimiento entre `.apply()` y las operaciones vectorizadas.

## 💡 Introducción

Imagina que `.apply()` es un puente que conecta el poder de las funciones de Python con la estructura de los DataFrames de pandas. En esencia, `.apply()` actúa como un bucle `for` optimizado que recorre cada elemento (o cada fila/columna) y le aplica una transformación específica.

Es la herramienta ideal cuando la lógica de transformación es demasiado compleja para expresarse con operadores matemáticos simples.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración del dataset
file_path = Path('players_20.csv')
df = pd.read_csv(file_path)
df.set_index('short_name', inplace=True)
df = df[['long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club']]

df.head()

## 🛠️ ¿Cómo funciona `.apply()`?

Podemos usar `.apply()` en dos niveles diferentes: sobre una **Serie** (una sola columna) o sobre un **DataFrame** (el conjunto completo).

### 🌉 Puente Pedagógico: La Metáfora de la Fábrica
Imagina una cinta transportadora de datos:
1. **Entrada**: Un dato (o una fila) entra en la "máquina" (tu función).
2. **Proceso**: La función procesa el dato siguiendo tus reglas.
3. **Salida**: El dato transformado sale y se coloca en el resultado.

```
   [ Dato Original ]  -->  [ f(x) : Tu Función ]  -->  [ Dato Transformado ]
          ^                                                 |
          |__________________(Repetir para cada elemento)___|
```

### 1. Aplicar a una Serie (Columna Única)

Cuando aplicamos `.apply()` a una sola columna, la función recibe como entrada cada valor individual de esa columna.

In [ ]:
# Ejemplo: Calcular la raíz cuadrada de la edad de los jugadores
# Usamos una función de numpy para demostrar la compatibilidad
raices_edad = df['age'].apply(np.sqrt)

print("Primeras 5 raíces cuadradas de la edad:")
print(raices_edad.head())

### 2. Aplicar a un DataFrame (Operaciones entre Columnas)

A veces, la transformación depende de **varias columnas a la vez**. Para esto, aplicamos la función al DataFrame completo y especificamos la dirección con `axis`.

#### El parámetro `axis=1`
Para que la función reciba una **fila completa** (como un objeto), debemos usar `axis=1`. Esto nos permite acceder a diferentes columnas de esa fila usando sus nombres.

In [ ]:
# Definimos una función personalizada para calcular el IMC (Índice de Masa Corporal)
def calcular_imc(fila: pd.Series) -> float:
    # IMC = peso (kg) / (altura (m)^2)
    peso = fila['weight_kg']
    altura_m = fila['height_cm'] / 100
    return peso / (altura_m ** 2)

# Aplicamos la función a lo largo del eje 1 (filas)
df['IMC'] = df.apply(calcular_imc, axis=1)

df[['long_name', 'height_cm', 'weight_kg', 'IMC']].head()

### ⚠️ Nota sobre el Rendimiento

Aunque `.apply()` es extremadamente flexible, es esencialmente un bucle `for` disfrazado. Para operaciones simples, siempre es preferible usar **operaciones vectorizadas**.

**Ejemplo de optimización:**
En lugar de `df.apply(lambda r: r['weight_kg'] / (r['height_cm']/100)**2, axis=1)`,
es mucho más rápido hacer: `df['weight_kg'] / (df['height_cm']/100)**2`.

## 📝 Ejercicios de Práctica

1. **Formateo de Texto**: Crea una función que reciba el nombre largo (`long_name`) y devuelva solo la primera letra en mayúscula seguida de un punto (ej. "Lionel Andrés..." $ightarrow$ "L."). Aplícala a la columna `long_name`.
2. **Categorización de Edad**: Crea una función que clasifique a los jugadores en "Joven" (< 23), "Prime" (23-30) y "Veterano" (> 30). Aplícala al DataFrame para crear una nueva columna llamada `categoria_edad`.
3. **Cálculo de Área**: Crea una función que calcule el "volumen corporal" simplificado multiplicando altura $	imes$ peso. Aplícala usando `.apply(..., axis=1)`.

## 📋 Resumen Rápido

| Aplicación | Objeto | Entrada de la Función | Ejemplo de Uso |
| :--- | :--- | :--- | :--- |
| **Serie** | `df['col'].apply(f)` | Un valor individual | `df['age'].apply(np.sqrt)` |
| **DataFrame** | `df.apply(f, axis=1)` | Una fila completa | `df.apply(mi_func, axis=1)` |
| **DataFrame** | `df.apply(f, axis=0)` | Una columna completa | `df.apply(np.sum, axis=0)` |